In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

In [ ]:
import zipfile
import os
os.makedirs("/content/dataset", exist_ok=True)
zip_path = "/content/food_dataset.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")
print("done")
print(os.listdir("/content/dataset"))

done
['food_dataset']


In [ ]:
dataset_path = "/content/dataset/food_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
train_datagen = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    validation_split=0.2
)

In [ ]:
train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 2416 images belonging to 11 classes.
Found 596 images belonging to 11 classes.


In [ ]:
print(train_data.class_indices)
print("Số class:", train_data.num_classes)

{'ca_hu_kho': 0, 'canh_chua_co_ca': 1, 'canh_chua_khong_ca': 2, 'canh_rau': 3, 'com_trang': 4, 'dau_hu_sot_ca': 5, 'rau_xao': 6, 'suon_nuong': 7, 'thit_kho_khong_trung': 8, 'thit_kho_trung': 9, 'trung_chien': 10}
Số class: 11


In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(train_data.num_classes, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "model_1.keras",
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.6734 - loss: 0.9906 - val_accuracy: 0.8775 - val_loss: 0.4032
Epoch 2/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 650ms/step - accuracy: 0.8647 - loss: 0.4104 - val_accuracy: 0.8993 - val_loss: 0.2962
Epoch 3/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 640ms/step - accuracy: 0.8849 - loss: 0.3416 - val_accuracy: 0.8876 - val_loss: 0.2815
Epoch 4/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 645ms/step - accuracy: 0.8949 - loss: 0.2952 - val_accuracy: 0.8943 - val_loss: 0.2561
Epoch 5/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 650ms/step - accuracy: 0.9139 - loss: 0.2422 - val_accuracy: 0.8993 - val_loss: 0.2536
Epoch 6/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 649ms/step - accuracy: 0.9201 - loss: 0.2262 - val_accuracy: 0.9077 - val_loss: 0.2694
Epoch 7/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 50s 659ms/step - accuracy: 0.9263 - loss: 0.2136 - val_accuracy: 0.8977 - val_loss: 0.2592
Epoch 8/15
76/76 ━━━━━━━━━━━━━━━━━━━━ 81s 644ms/step - accuracy: 0.9325 - loss: 0.1948 - val_accurac

In [ ]:
import tensorflow as tf

model_1 = tf.keras.models.load_model("model_1.keras")

model_1.save("food_model.keras")

from google.colab import files
files.download("food_model.keras")

print("done")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

done


In [ ]:
import tensorflow as tf
from google.colab import files

model = tf.keras.models.load_model("food_model.keras", compile=False)
model.save("food_model.h5")

files.download("food_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>